In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [2]:
import random

import torch
import numpy as np
from torch.utils.data import Dataset
from torchvision.io import read_image

class MassHumanBurns(Dataset):
    _views_per_burn = 4
    def __init__(self, root_dir, train_mode = False,  return_images=True, return_full_masks=True, return_burn_masks=True):
        self.root_dir = root_dir
        self.train_mode = train_mode
        if train_mode:
            self._pairs_per_burn = 1
        else:
            self._pairs_per_burn = self._views_per_burn ** 2
        self.labels = torch.tensor(np.load(os.path.join(root_dir, "labels.npy")), dtype=torch.float32)
        self.return_images = return_images
        self.return_full_masks = return_full_masks
        self.return_burn_masks = return_burn_masks
        self._len = self.labels.numel() * self._pairs_per_burn
        self._burn_levels = self.labels.shape[1]
        self._pairs_per_human = self._burn_levels * self._pairs_per_burn

    def __len__(self):
        return self._len

    def __getitem__(self, idx):
        human_idx = idx // self._pairs_per_human
        burn_idx = idx % self._pairs_per_human // self._pairs_per_burn
        if self.train_mode:
            front_view_idx = random.randint(0, self._views_per_burn - 1)
            back_view_idx = random.randint(0, self._views_per_burn - 1)
        else:
            front_view_idx = idx % self._pairs_per_human % self._pairs_per_burn // self._views_per_burn
            back_view_idx = idx % self._pairs_per_human % self._pairs_per_burn % self._views_per_burn
        res = {"label": self.labels[human_idx, burn_idx]}
        if self.return_images:
            res["images"] = (
                read_image(os.path.join(self.root_dir, "images", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.jpg")), 
                read_image(os.path.join(self.root_dir, "images", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.jpg"))
            )
        if self.return_full_masks:
            res["full_masks"] = (
                read_image(os.path.join(self.root_dir, "full_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                read_image(os.path.join(self.root_dir, "full_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        if self.return_burn_masks:
            res["burn_masks"] = (
                read_image(os.path.join(self.root_dir, "burn_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                read_image(os.path.join(self.root_dir, "burn_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        return res


def pad_to_square(image, size=512):
    _, h, w = image.shape
    l = (512-w) // 2
    t = (512-h) // 2
    r = 512 - w - l
    b = 512 - h - t
    return torch.nn.functional.pad(image, (l, r, t, b))

def data_collator(inputs,*kwargs):
    with torch.no_grad():
        front_full_mask = []
        back_full_mask = []
        front_burn_mask = []
        back_burn_mask = []
        labels = []

        for input in inputs:
            front_full_mask.append(pad_to_square(input["full_masks"][0]))
            back_full_mask.append(pad_to_square(input["full_masks"][1]))
            front_burn_mask.append(pad_to_square(input["burn_masks"][0]))
            back_burn_mask.append(pad_to_square(input["burn_masks"][1]))
            labels.append(input["label"])

        front_full_mask = torch.stack(front_full_mask, dim=0)
        back_full_mask = torch.stack(back_full_mask, dim=0)
        front_burn_mask = torch.stack(front_burn_mask, dim=0)
        back_burn_mask = torch.stack(back_burn_mask, dim=0)
        labels = torch.stack(labels, dim=0)
        return {
            "front_full_mask": front_full_mask,
            "back_full_mask": back_full_mask,
            "front_burn_mask": front_burn_mask,
            "back_burn_mask": back_burn_mask,
            "labels": labels
        }

In [3]:
from torch.utils.data import Subset

train_set = MassHumanBurns("mass_human_burns", train_mode=True, return_images=False)
test_set = MassHumanBurns("mass_human_burns", return_images=False)

train_indices = torch.tensor([list(range(i*625*train_set._pairs_per_human,(i*625+500)*train_set._pairs_per_human)) for i in range(8)]).flatten()
test_indices = torch.tensor([list(range((i*625+500)*test_set._pairs_per_human,(i*625+625)*test_set._pairs_per_human)) for i in range(8)]).flatten()

train_set = Subset(train_set, train_indices)
test_set = Subset(test_set, test_indices)


In [4]:
from typing import Callable, Optional, Union
from torchvision.models.resnet import BasicBlock, Bottleneck, conv1x1

class ResNetBackbone(torch.nn.Module):
    def __init__(
        self,
        block: type[Union[BasicBlock, Bottleneck]],
        layers: list[int],
        zero_init_residual: bool = False,
        groups: int = 1,
        width_per_group: int = 64,
        replace_stride_with_dilation: Optional[list[bool]] = None,
        norm_layer: Optional[Callable[..., torch.nn.Module]] = None,
    ) -> None:
        super().__init__()
        if norm_layer is None:
            norm_layer = torch.nn.BatchNorm2d
        self._norm_layer = norm_layer

        self.inplanes = 64
        self.dilation = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        if len(replace_stride_with_dilation) != 3:
            raise ValueError(
                "replace_stride_with_dilation should be None "
                f"or a 3-element tuple, got {replace_stride_with_dilation}"
            )
        self.groups = groups
        self.base_width = width_per_group
        self.conv1 = torch.nn.Conv2d(1, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = norm_layer(self.inplanes)
        self.relu = torch.nn.ReLU(inplace=True)
        self.maxpool = torch.nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0])
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2, dilate=replace_stride_with_dilation[1])
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2, dilate=replace_stride_with_dilation[2])
        self.avgpool = torch.nn.AdaptiveAvgPool2d((1, 1))

        for m in self.modules():
            if isinstance(m, torch.nn.Conv2d):
                torch.nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (torch.nn.BatchNorm2d, torch.nn.GroupNorm)):
                torch.nn.init.constant_(m.weight, 1)
                torch.nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck) and m.bn3.weight is not None:
                    torch.nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock) and m.bn2.weight is not None:
                    torch.nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(
        self,
        block: type[Union[BasicBlock, Bottleneck]],
        planes: int,
        blocks: int,
        stride: int = 1,
        dilate: bool = False,
    ) -> torch.nn.Sequential:
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = torch.nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(
            block(
                self.inplanes, planes, stride, downsample, self.groups, self.base_width, previous_dilation, norm_layer
            )
        )
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(
                block(
                    self.inplanes,
                    planes,
                    groups=self.groups,
                    base_width=self.base_width,
                    dilation=self.dilation,
                    norm_layer=norm_layer,
                )
            )

        return torch.nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        return x


In [5]:
class BurnAreaNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone_a = ResNetBackbone(BasicBlock, [2, 2, 2, 2])
        self.backbone_b = ResNetBackbone(BasicBlock, [2, 2, 2, 2])
        self.head = torch.nn.Sequential(
            torch.nn.Linear(1024, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, 1),
            torch.nn.Sigmoid()
        )

    def forward(self, front_full_mask, back_full_mask, front_burn_mask, back_burn_mask):
        features = torch.cat([self.backbone_a(front_full_mask), self.backbone_a(back_full_mask)], dim=-1) - torch.cat([self.backbone_b(front_burn_mask), self.backbone_b(back_burn_mask)], dim=-1)
        return self.head(features)

In [6]:
from torchvision.transforms import v2 as T

transform = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.uint8, scale=True),
    T.Resize((224, 224)),
    T.ToDtype(torch.float32,scale=True)
])

class HFCompatibleModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, front_full_mask, back_full_mask, front_burn_mask, back_burn_mask, labels=None, **kwargs):
        pre = self.model(transform(front_full_mask), transform(back_full_mask), transform(front_burn_mask), transform(back_burn_mask))[:, 0]
        if labels is not None:
            loss = (pre - labels).abs().mean()
            return {"loss": loss, "logits": pre}
        else:
            return {"logits": pre}

In [7]:
import time
import transformers

trainer_args = transformers.TrainingArguments(
    num_train_epochs=480,
    output_dir="./logs/" + time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime()),
    dataloader_num_workers=8,
    per_device_train_batch_size=128,
    weight_decay=1e-3,
    logging_strategy="epoch",
    remove_unused_columns=False,
    save_strategy="epoch",
    save_total_limit=1,
    metric_for_best_model="loss",
    greater_is_better=False,
    bf16=True,
    bf16_full_eval=True,
)

compatible_model = HFCompatibleModel(BurnAreaNet()).cuda()

In [8]:
# trainer = transformers.Trainer(
#     model = compatible_model,
#     train_dataset=train_set,
#     data_collator=data_collator,
#     args=trainer_args,
# )

# trainer.train()

In [9]:
from safetensors.torch import load_file

compatible_model.load_state_dict(load_file("./model.safetensors"))

<All keys matched successfully>

In [10]:
from torch.utils.data import DataLoader
from tqdm import tqdm

test_dataloader = DataLoader(test_set, batch_size=128, num_workers=8, collate_fn=data_collator)
compatible_model.eval()
pre_list = []
label_list = []
with torch.no_grad():
    for batch in tqdm(test_dataloader):
        output = compatible_model(batch["front_full_mask"].cuda(), batch["back_full_mask"].cuda(), batch["front_burn_mask"].cuda(), batch["back_burn_mask"].cuda())
        pre_list.append(output["logits"].cpu())
        label_list.append(batch["labels"])

pre_list = torch.cat(pre_list).reshape(1000,8,16)
label_list = torch.cat(label_list).reshape(1000,8,16)

100%|██████████| 1000/1000 [02:26<00:00,  6.81it/s]


In [11]:
from sklearn.metrics import r2_score
print("R2 Score:", r2_score(label_list.flatten(), pre_list.flatten()))

R2 Score: 0.9990158081054688


In [12]:
triclass_label = torch.zeros_like(label_list)
triclass_label[label_list < 0.05] = 0
triclass_label[(label_list >= 0.05) & (label_list < 0.2)] = 1
triclass_label[label_list >= 0.2] = 2
triclass_pre = torch.zeros_like(pre_list)
triclass_pre[pre_list < 0.05] = 0
triclass_pre[(pre_list >= 0.05) & (pre_list < 0.2)] = 1
triclass_pre[pre_list >= 0.2] = 2
from sklearn.metrics import classification_report
print(classification_report(triclass_label.flatten(), triclass_pre.flatten(), digits=3))

              precision    recall  f1-score   support

         0.0      0.972     0.971     0.971     15600
         1.0      0.974     0.976     0.975     32640
         2.0      0.996     0.995     0.995     79760

    accuracy                          0.987    128000
   macro avg      0.980     0.981     0.981    128000
weighted avg      0.987     0.987     0.987    128000



In [13]:
mae = (pre_list-label_list).abs()
print(f"MAE:{mae.mean().item()*100:.2f}, STD:{torch.std(mae).item()*100:.2f}")

MAE:0.59, STD:0.52


In [14]:
mre = (pre_list-label_list).abs()/(label_list)
print(f"MRE:{mre.mean().item()*100:.2f}, STD:{torch.std(mre).item()*100:.2f}")

MRE:3.64, STD:6.62


In [15]:
import pandas as pd
import pingouin as pg

reshaped_pre = pre_list.view(-1, test_set.dataset._pairs_per_burn)
n_subjects, n_raters = reshaped_pre.shape

long_data_list = []
for i in range(n_raters):
    rater_name = f'Observer_{i+1}'
    temp_df = pd.DataFrame({
        'subject': torch.arange(n_subjects),
        'rater': rater_name,
        'rating': reshaped_pre[:, i] * 100
    })
    long_data_list.append(temp_df)

long_data = pd.concat(long_data_list, ignore_index=True)
pg.intraclass_corr(
    data=long_data,
    targets='subject',
    raters='rater',
    ratings='rating'
)

,Type,Description,ICC,F,df1,df2,pval,CI95
0,ICC1,Single raters absolute,0.999769,69352.096874,7999,120000,0.0,"[1.0, 1.0]"
1,ICC2,Single random raters,0.999769,69349.642548,7999,119985,0.0,"[1.0, 1.0]"
2,ICC3,Single fixed raters,0.999769,69349.642548,7999,119985,0.0,"[1.0, 1.0]"
3,ICC1k,Average raters absolute,0.999986,69352.096874,7999,120000,0.0,"[1.0, 1.0]"
4,ICC2k,Average random raters,0.999986,69349.642548,7999,119985,0.0,"[1.0, 1.0]"
5,ICC3k,Average fixed raters,0.999986,69349.642548,7999,119985,0.0,"[1.0, 1.0]"


In [16]:
baseline_pre = torch.load("baseline_pre.pt")
baseline_mae = (baseline_pre - label_list).abs()
baseline_mre = ((baseline_pre - label_list).abs()/label_list)
print(pg.ttest((baseline_mae.view(-1)*100).numpy().tolist(), (mae.view(-1)*100).numpy().tolist(), paired=True))
print(pg.ttest((baseline_mre.view(-1)*100).numpy().tolist(), (mre.view(-1)*100).numpy().tolist(), paired=True))

                T     dof alternative  p_val          CI95   cohen_d  power  \
T_test  280.79835  127999   two-sided    0.0  [1.12, 1.14]  1.021502    1.0   

       BF10  
T_test  inf  
                 T     dof alternative  p_val          CI95   cohen_d  power  \
T_test  194.642285  127999   two-sided    0.0  [4.67, 4.76]  0.588252    1.0   

       BF10  
T_test  inf  


In [17]:
for i in range(8):
    print(f"Burn Level {i+1}:", end=" ")
    print(f"MAE: {mae[:, i].mean().item()*100:.2f}±{torch.std(mae[:, i]).item()*100:.2f}, ", end=" / ")
    print(f"{baseline_mae[:, i].mean().item()*100:.2f}±{torch.std(baseline_mae[:, i]).item()*100:.2f}, ", end="")
    print(f"MRE: {mre[:, i].mean().item()*100:.2f}±{torch.std(mre[:, i]).item()*100:.2f}", end=" / ")
    print(f"{baseline_mre[:, i].mean().item()*100:.2f}±{torch.std(baseline_mre[:, i]).item()*100:.2f}")

Burn Level 1: MAE: 0.22±0.20,  / 0.45±0.42, MRE: 11.87±15.05 / 19.77±16.20
Burn Level 2: MAE: 0.39±0.30,  / 0.91±0.71, MRE: 5.20±4.07 / 12.22±9.40
Burn Level 3: MAE: 0.52±0.40,  / 1.33±1.01, MRE: 3.58±2.83 / 9.17±6.95
Burn Level 4: MAE: 0.59±0.47,  / 1.74±1.28, MRE: 2.43±1.92 / 7.09±5.21
Burn Level 5: MAE: 0.68±0.53,  / 2.02±1.50, MRE: 1.97±1.55 / 5.90±4.37
Burn Level 6: MAE: 0.74±0.57,  / 2.24±1.64, MRE: 1.68±1.30 / 5.08±3.74
Burn Level 7: MAE: 0.81±0.62,  / 2.50±1.70, MRE: 1.40±1.07 / 4.33±2.95
Burn Level 8: MAE: 0.78±0.61,  / 2.59±1.49, MRE: 0.99±0.78 / 3.30±2.00


In [18]:
import json

with open("mass_human_burns/gender.json", "r") as f:
    gender_data = json.load(f)

gender_mask = torch.zeros(5000, dtype=bool)
gender_mask[torch.tensor(gender_data["male"])]=1

test_human_indices = (test_indices // test_set.dataset._pairs_per_human).view([1000,8,16])[:,0,0]
test_gender_mask = gender_mask[test_human_indices]

print(f"Number of males: {test_gender_mask.sum().item()}, Number of females: {(~test_gender_mask).sum().item()}")

print(f"Male:", end=" ")
print(f"MAE: {mae[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(mae[test_gender_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mae[test_gender_mask==1]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(mre[test_gender_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mre[test_gender_mask==1]).item()*100:.2f}")
print(f"Female:", end=" ")
print(f"MAE: {mae[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(mae[test_gender_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mae[test_gender_mask==0]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(mre[test_gender_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mre[test_gender_mask==0]).item()*100:.2f}")

print(pg.ttest((mae[test_gender_mask==0].view(-1)*100).numpy().tolist(), (mae[test_gender_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[test_gender_mask==0].view(-1)*100).numpy().tolist(), (mre[test_gender_mask==1].view(-1)*100).numpy().tolist()))

Number of males: 493, Number of females: 507
Male: MAE: 0.59±0.52 / 1.78±1.53, MRE: 3.65±7.19 / 8.52±9.33
Female: MAE: 0.59±0.52 / 1.67±1.43, MRE: 3.63±6.03 / 8.20±9.08
               T            dof alternative     p_val         CI95   cohen_d  \
T_test  2.105482  127923.645477   two-sided  0.035251  [0.0, 0.01]  0.011771   

           power   BF10  
T_test  0.557821  0.058  
               T            dof alternative     p_val           CI95  \
T_test -0.493003  122990.372282   two-sided  0.622011  [-0.09, 0.05]   

         cohen_d     power   BF10  
T_test  0.002763  0.078417  0.007  


In [19]:
age_mask = torch.zeros(1000, dtype=bool)
age_mask[:500] = 1

print(f"Adult:", end=" ")
print(f"MAE: {mae[age_mask==1].mean().item()*100:.2f}±{torch.std(mae[age_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[age_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mae[age_mask==1]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[age_mask==1].mean().item()*100:.2f}±{torch.std(mre[age_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[age_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mre[age_mask==1]).item()*100:.2f}")
print(f"Child:", end=" ")
print(f"MAE: {mae[age_mask==0].mean().item()*100:.2f}±{torch.std(mae[age_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[age_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mae[age_mask==0]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[age_mask==0].mean().item()*100:.2f}±{torch.std(mre[age_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[age_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mre[age_mask==0]).item()*100:.2f}")

print(pg.ttest((mae[age_mask==0].view(-1)*100).numpy().tolist(), (mae[age_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[age_mask==0].view(-1)*100).numpy().tolist(), (mre[age_mask==1].view(-1)*100).numpy().tolist()))

Adult: MAE: 0.57±0.50 / 1.73±1.50, MRE: 3.47±5.62 / 8.39±9.37
Child: MAE: 0.61±0.54 / 1.71±1.45, MRE: 3.81±7.49 / 8.32±9.03
                T     dof alternative         p_val          CI95   cohen_d  \
T_test  10.829394  127998   two-sided  2.567162e-27  [0.03, 0.04]  0.060538   

        power       BF10  
T_test    1.0  1.784e+23  
               T     dof alternative         p_val          CI95   cohen_d  \
T_test  9.109706  127998   two-sided  8.375056e-20  [0.26, 0.41]  0.050925   

        power       BF10  
T_test    1.0  6.491e+15  


In [20]:
for i in range(4):
    pose_mask = torch.zeros(1000, dtype=bool)
    pose_mask[i*125:(i+1)*125] = 1 # Adult
    pose_mask[i*125+500:(i+1)*125+500] = 1 # Child
    print(f"Pose {i+1}:", end=" ")
    print(f"MAE: {mae[pose_mask==1].mean().item()*100:.2f}±{torch.std(mae[pose_mask==1]).item()*100:.2f}", end=" / ")
    print(f"{baseline_mae[pose_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mae[pose_mask==1]).item()*100:.2f}", end=" / ")
    print(f"MRE: {mre[pose_mask==1].mean().item()*100:.2f}±{torch.std(mre[pose_mask==1]).item()*100:.2f}", end=" / ")
    print(f"{baseline_mre[pose_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mre[pose_mask==1]).item()*100:.2f}")

Pose 1: MAE: 0.64±0.55 / 1.64±1.40 / MRE: 3.91±6.43 / 8.23±9.42
Pose 2: MAE: 0.59±0.51 / 1.73±1.50 / MRE: 3.70±7.91 / 8.47±9.50
Pose 3: MAE: 0.58±0.52 / 1.70±1.46 / MRE: 3.63±6.47 / 8.10±8.67
Pose 4: MAE: 0.55±0.50 / 1.81±1.54 / MRE: 3.31±5.42 / 8.63±9.20


In [21]:
pose_1_mask = torch.zeros(1000, dtype=bool)
pose_1_mask[:125] = 1
pose_1_mask[500:625] = 1
pose_4_mask = torch.zeros(1000, dtype=bool)
pose_4_mask[375:500] = 1
pose_4_mask[875:1000] = 1

print(pg.ttest((mae[pose_1_mask==1].view(-1)*100).numpy().tolist(), (mae[pose_4_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[pose_1_mask==1].view(-1)*100).numpy().tolist(), (mre[pose_4_mask==1].view(-1)*100).numpy().tolist()))

                T    dof alternative         p_val          CI95   cohen_d  \
T_test  20.746673  63998   two-sided  2.709344e-95  [0.08, 0.09]  0.164017   

        power       BF10  
T_test    1.0  1.207e+91  
                T    dof alternative         p_val          CI95   cohen_d  \
T_test  12.809058  63998   two-sided  1.622497e-37  [0.51, 0.69]  0.101264   

        power       BF10  
T_test    1.0  3.345e+33  
